In [2]:
# HMM POS Tagger
# Universal Dependencies - English EWT

import pandas as pd
import numpy as np
from collections import defaultdict

# 1. Load the Universal Dependencies English EWT dataset
def load_conllu(filename):
    sentences = []
    sentence = []

    with open(filename, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            # Blank line indicates end of sentence
            if line == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue

            # Ignore comments
            if line.startswith("#"):
                continue

            columns = line.split("\t")

            if len(columns) != 10:
                continue

            # Ignore multi-word tokens and empty nodes
            if "-" in columns[0] or "." in columns[0]:
                continue

            word = columns[1]
            tag = columns[3]

            if tag != "_":
                sentence.append((word, tag))

    if sentence:
        sentences.append(sentence)

    return sentences

train_data = load_conllu(
    "en_ewt-ud-train.conllu"
)

test_data = load_conllu(
    "en_ewt-ud-test.conllu"
)

print("Training Sentences:", len(train_data))
print("Testing Sentences:", len(test_data))

# 2. Extract words and their POS tags
vocabulary = set()
tags = set()

for sentence in train_data:

    for word, tag in sentence:

        vocabulary.add(word.lower())
        tags.add(tag)

vocabulary = list(vocabulary)
tags = list(tags)

word_to_index = {
    word: i
    for i, word in enumerate(vocabulary)
}

tag_to_index = {
    tag: i
    for i, tag in enumerate(tags)
}

index_to_tag = {
    i: tag
    for tag, i in tag_to_index.items()
}


V = len(vocabulary)
T = len(tags)

print("Vocabulary Size:", V)
print("Number of POS Tags:", T)

# 3. Calculate Transition Probabilities
transition_count = np.ones((T, T))

for sentence in train_data:

    for i in range(1, len(sentence)):

        previous_tag = sentence[i - 1][1]
        current_tag = sentence[i][1]

        previous_index = tag_to_index[previous_tag]
        current_index = tag_to_index[current_tag]

        transition_count[
            previous_index,
            current_index
        ] += 1

transition_probability = (
    transition_count /
    transition_count.sum(
        axis=1,
        keepdims=True
    )
)

# 4. Calculate Emission Probabilities
emission_count = np.ones((T, V))

for sentence in train_data:

    for word, tag in sentence:

        word = word.lower()

        tag_index = tag_to_index[tag]

        if word in word_to_index:

            word_index = word_to_index[word]

            emission_count[
                tag_index,
                word_index
            ] += 1

emission_probability = (
    emission_count /
    emission_count.sum(
        axis=1,
        keepdims=True
    )
)

# Convert probabilities to log probabilities
log_transition = np.log(
    transition_probability
)

log_emission = np.log(
    emission_probability
)

# 5. Viterbi Algorithm using HMM
def viterbi(words):

    n = len(words)

    dp = np.full(
        (T, n),
        -np.inf
    )
    backpointer = np.zeros(
        (T, n),
        dtype=int
    )

    # First word
    word = words[0].lower()

    if word in word_to_index:

        word_index = word_to_index[word]

        emission = log_emission[
            :,
            word_index
        ]
    else:
        emission = np.log(
            np.ones(T) * 1e-10
        )
    # Equal initial probability
    initial_probability = np.ones(T) / T

    dp[:, 0] = (
        np.log(initial_probability)
        + emission
    )
    # Remaining words
    for i in range(1, n):

        word = words[i].lower()

        if word in word_to_index:

            word_index = word_to_index[word]

            emission = log_emission[
                :,
                word_index
            ]
        else:

            emission = np.log(
                np.ones(T) * 1e-10
            )
        scores = (
            dp[:, i - 1][:, None]
            + log_transition
        )
        backpointer[:, i] = np.argmax(
            scores,
            axis=0
        )
        dp[:, i] = (
            np.max(
                scores,
                axis=0
            )
            + emission
        )
    # Backtracking

    best_tag = np.argmax(
        dp[:, -1]
    )

    result = [best_tag]

    for i in range(n - 1, 0, -1):

        best_tag = backpointer[
            best_tag,
            i
        ]
        result.append(best_tag)

    result.reverse()

    return [
        index_to_tag[i]
        for i in result
    ]

# 6 & 7. Accept a sentence and predict POS tags
sentence = input(
    "\nEnter a sentence: "
)

words = sentence.split()
predicted_tags = viterbi(words)

print("\nPredicted POS Tags:")
print("-------------------")

for word, tag in zip(
    words,
    predicted_tags
):

    print(
        word,
        "-->",
        tag
    )

# 8. Compare predicted tags with actual tags
actual = []
predicted = []

for sentence in test_data:

    words = [
        word
        for word, tag in sentence
    ]

    actual_tags = [
        tag
        for word, tag in sentence
    ]

    predicted_tags = viterbi(words)

    actual.extend(actual_tags)
    predicted.extend(predicted_tags)

# 9. Calculate Accuracy and Evaluation Report
correct = sum(
    a == p
    for a, p in zip(
        actual,
        predicted
    )
)

total = len(actual)

accuracy = correct / total


print("\nEvaluation Report")
print("-----------------")

print(
    "Total Test Words:",
    total
)

print(
    "Correct Predictions:",
    correct
)

print(
    "Incorrect Predictions:",
    total - correct
)

print(
    "POS Tagging Accuracy:",
    round(
        accuracy * 100,
        2
    ),
    "%"
)

Training Sentences: 12544
Testing Sentences: 2077
Vocabulary Size: 16654
Number of POS Tags: 17



Enter a sentence:  The student reads a book



Predicted POS Tags:
-------------------
The --> DET
student --> NOUN
reads --> ADP
a --> DET
book --> NOUN

Evaluation Report
-----------------
Total Test Words: 25094
Correct Predictions: 21413
Incorrect Predictions: 3681
POS Tagging Accuracy: 85.33 %
